# Cleaning the catalogue

Turn the raw CHIME/FRB Catalog 1 table into a tidy, scaled feature
table ready for outlier detection. Every decision is written into the cells
as comments so nothing happens off-screen. Run the cells top to bottom.

## Setup: load the shared reader and the raw catalogue

In [1]:
# Make the shared code in the project root's src/ importable. The reader lives in
# src/frb_anomaly/data.py so any notebook can reuse it (this is the 'module' habit).
import sys
from pathlib import Path

# Notebook now lives at Phase 1/notebooks/, so two .parent hops reach the project root
# where src/, data/, etc live.
PROJECT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT / 'src'))

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from frb_anomaly import data

fields, df = data.load_catalog1()
print(f'Loaded {len(df)} sub-burst rows, {df.shape[1]} columns')

Loaded 600 sub-burst rows, 59 columns


## Drop CHIME's excluded bursts

In [2]:
# CHIME sets Flag == 1 for bursts they excluded from their own analysis
# (unreliable detections: far sidelobe, non-nominal operation, etc.).
# We honour that flag. Keeping them would give us 'anomalies' that are really data faults.
before = len(df)
df = df[df['Flag'] == 0].copy()
print(f'Dropped {before - len(df)} excluded bursts; {len(df)} remain')

Dropped 42 excluded bursts; 558 remain


## Label repeaters (do not drop them yet)

In [3]:
# RpName holds the repeating source's name, or '-9999' for a one-off burst.
# We only LABEL here. Option C needs both groups:
#   Pass 1 (sanity check) uses everyone, with this label.
#   Pass 2 (real hunt) will keep only the one-offs.
df['is_repeater'] = df['RpName'].astype(str).str.strip() != '-9999'
print(df['is_repeater'].value_counts().rename({False: 'one-off', True: 'repeater'}))

is_repeater
one-off     468
repeater     90
Name: count, dtype: int64


## Choose the feature columns for the burst's shape

In [4]:
# These describe what the burst looks like. We deliberately leave out sky position,
# arrival time, exposure and fit-quality, none of which are about the burst's shape.
# The original CHIME name is in the comment, for when we move to Catalog 2 later.
FEATURES = {
    'DMfitb':    'dispersion measure (dm_fitb)',
    'Widthfitb': 'width (width_fitb)',
    'Scat':      'scattering time (scat_time) -- ~half are upper limits, least trustworthy',
    'Flux':      'peak flux (lower limit)',
    'Fluence':   'fluence (lower limit)',
    'SpInd':     'spectral index (sp_idx)',
    'spRun':     'spectral running (sp_run)',
    'Fpk':       'peak frequency (peak_freq)',
}

# Bandwidth is not a single column; build it from the high/low detection frequencies.
df['Bandwidth'] = df['B_Freq'] - df['b_Freq']
FEATURES['Bandwidth'] = 'detection bandwidth (high_freq - low_freq)'

feature_cols = list(FEATURES)
print(f'Using {len(feature_cols)} features:')
for c, desc in FEATURES.items():
    print(f'  {c:11s} {desc}')

Using 9 features:
  DMfitb      dispersion measure (dm_fitb)
  Widthfitb   width (width_fitb)
  Scat        scattering time (scat_time) -- ~half are upper limits, least trustworthy
  Flux        peak flux (lower limit)
  Fluence     fluence (lower limit)
  SpInd       spectral index (sp_idx)
  spRun       spectral running (sp_run)
  Fpk         peak frequency (peak_freq)
  Bandwidth   detection bandwidth (high_freq - low_freq)


## Example data

In [5]:
# Worth seeing the spread first. Note how DM is in the hundreds-to-thousands while
# scattering is ~0.001, and spectral running has some very extreme values. That huge
# range across columns is exactly why we transform and rescale.
df[feature_cols].describe().round(3).T

,count,mean,std,min,25%,50%,75%,max
DMfitb,558.0,627.728,419.530,103.396,348.836,514.985,778.159,3038.058
Widthfitb,558.0,0.002,0.002,0.000,0.001,0.001,0.002,0.026
Scat,558.0,0.004,0.008,0.000,0.001,0.001,0.003,0.090
Flux,558.0,1.784,2.975,0.000,0.542,0.915,1.810,41.000
Fluence,558.0,8.325,12.037,0.000,2.200,4.165,9.100,95.000
SpInd,558.0,14.497,21.303,-9.400,0.400,5.115,21.075,99.000
spRun,558.0,-36.717,65.753,-910.000,-51.850,-11.050,-4.025,10.300
Fpk,558.0,512.902,112.434,400.200,422.600,477.800,582.750,800.200
Bandwidth,558.0,289.371,107.542,42.400,195.825,307.050,400.000,400.000


## Data cleaning  (limits & missing values)

In [6]:
# Limits: flux & fluence are lower limits, scattering is often an upper limit.
# We keep the values but stay aware they are boundaries, not exact (noted for the report).
# Missing values: rather than invent numbers to fill gaps, we keep only
# bursts that have a complete set of features, and report how many we lose.
feat = df[['Name', 'is_repeater', 'RpName'] + feature_cols].copy()

complete = feat.dropna(subset=feature_cols)
lost = len(feat) - len(complete)
print(f'{len(complete)} of {len(feat)} bursts have all {len(feature_cols)} features '
      f'({lost} dropped for missing values)')
for c in feature_cols:
    n = feat[c].isna().sum()
    if n:
        print(f'  {c:11s}: {n} missing')

558 of 558 bursts have all 9 features (0 dropped for missing values)


## Transform and rescale

In [7]:
# Several quantities span orders of magnitude, so log-scale them first.
# DM, width and scattering are strictly positive -> plain log spreads them out.
log_plain = ['DMfitb', 'Widthfitb', 'Scat']
# Flux and fluence can be ~0 -> log1p = log(1 + x) is safe at zero.
log_1p = ['Flux', 'Fluence']

transformed = complete.copy()
transformed[log_plain] = np.log(transformed[log_plain])
transformed[log_1p] = np.log1p(transformed[log_1p])

# Then put every feature on a common scale (mean 0, spread 1) so the outlier maths
# treats them fairly instead of mostly reacting to the big-numbered columns.
scaler = StandardScaler()
X = scaler.fit_transform(transformed[feature_cols])
scaled = pd.DataFrame(X, columns=feature_cols, index=transformed.index)

print('Each feature now has ~mean 0 and ~std 1:')
print(scaled.describe().loc[['mean', 'std']].round(2).T)

Each feature now has ~mean 0 and ~std 1:
           mean  std
DMfitb     -0.0  1.0
Widthfitb  -0.0  1.0
Scat        0.0  1.0
Flux       -0.0  1.0
Fluence     0.0  1.0
SpInd      -0.0  1.0
spRun       0.0  1.0
Fpk         0.0  1.0
Bandwidth   0.0  1.0


## Save the cleaned tables for the outlier step

In [8]:
# Save two files the next notebook will load:
#  - scaled features: what the outlier maths runs on
#  - raw features: original values, needed later to explain WHY a burst is an outlier
ids = complete[['Name', 'is_repeater', 'RpName']].reset_index(drop=True)
scaled_out = pd.concat([ids, scaled.reset_index(drop=True)], axis=1)
raw_out = pd.concat([ids, complete[feature_cols].reset_index(drop=True)], axis=1)

# Phase 1 outputs go into data/processed/phase_1/ (Phase 2 will write to phase_2/).
processed_dir = PROJECT / 'data' / 'processed' / 'phase_1'
processed_dir.mkdir(parents=True, exist_ok=True)
scaled_out.to_csv(processed_dir / 'catalog1_features_scaled.csv', index=False)
raw_out.to_csv(processed_dir / 'catalog1_features_raw.csv', index=False)

print('Saved:')
print('  data/processed/phase_1/catalog1_features_scaled.csv  (for the outlier maths)')
print('  data/processed/phase_1/catalog1_features_raw.csv     (original values, for interpreting outliers)')
scaled_out.head()

Saved:
  data/processed/phase_1/catalog1_features_scaled.csv  (for the outlier maths)
  data/processed/phase_1/catalog1_features_raw.csv     (original values, for interpreting outliers)


,Name,is_repeater,RpName,DMfitb,Widthfitb,Scat,Flux,Fluence,SpInd,spRun,Fpk,Bandwidth
0,FRB20180904A,False,-9999,-0.607669,-0.789727,-0.804483,1.384727,0.178044,-0.111663,0.206063,0.060520,0.211536
1,FRB20180906A,False,-9999,-0.508394,0.221567,0.098604,0.255236,-0.572264,-0.690496,0.580220,2.557567,1.029623
2,FRB20180906B,False,-9999,2.918080,1.044417,0.929283,-0.938590,-1.033623,-0.558943,0.389945,-0.566190,0.181754
3,FRB20180907D,False,-9999,1.690263,0.756859,0.468832,-0.332321,-0.023141,-0.549546,0.557387,2.557567,1.029623
4,FRB20180907A,False,-9999,0.861626,-0.251268,-1.391863,-0.351919,-0.540880,-0.789160,0.481277,-1.003284,-0.381321
